# 02 - Harmonisation between 2025 and 2026

Goal: find which questions can be compared between the two years, and how. The decisions are written in `docs/HARMONISATION.md` and coded in `src/labels.py` and `src/cleaning.py`.

In [1]:
import sys
sys.path.append("..")   # to import the src package of the project

import pandas as pd
from src.data import download_year
from src.config import RAW_FILES

pd.set_option("display.max_colwidth", 80)

In [2]:
download_year(2025)
download_year(2026)
r25 = pd.read_csv(RAW_FILES[2025]["survey_path"], encoding="utf-8-sig", low_memory=False)
r26 = pd.read_csv(RAW_FILES[2026]["survey_path"], encoding="utf-8-sig", low_memory=False)
print("2025:", r25.shape, "| 2026:", r26.shape)

2025: (49191, 172) | 2026: (30903, 212)


## 1. Same question, same answers?

For each question we need, we compare the list of possible answers of the two years.

In [3]:
def compare_answers(column_2025, column_2026):
    a = set(r25[column_2025].dropna())
    b = set(r26[column_2026].dropna())
    return {"only in 2025": sorted(a - b), "only in 2026": sorted(b - a)}

pairs = [("Country", "Country"), ("Employment", "Employment"), ("OrgSize", "OrgSize"),
         ("RemoteWork", "WorkSetting"), ("Age", "Age"), ("EdLevel", "Education")]
for c25, c26 in pairs:
    diff = compare_answers(c25, c26)
    print(f"{c25} / {c26}: {len(diff['only in 2025'])} only in 2025, {len(diff['only in 2026'])} only in 2026")

Country / Country: 14 only in 2025, 6 only in 2026
Employment / Employment: 0 only in 2025, 0 only in 2026
OrgSize / OrgSize: 0 only in 2025, 0 only in 2026
RemoteWork / WorkSetting: 0 only in 2025, 0 only in 2026
Age / Age: 0 only in 2025, 1 only in 2026
EdLevel / Education: 3 only in 2025, 3 only in 2026


In [4]:
compare_answers("EdLevel", "Education")

{'only in 2025': ['Bachelor’s degree (B.A., B.S., B.Eng., etc.)',
  'Master’s degree (M.A., M.S., M.Eng., MBA, etc.)',
  'Other (please specify):'],
 'only in 2026': ["Bachelor's degree (B.A., B.S., B.Eng., etc.)",
  "Master's degree (M.A., M.S., M.Eng., MBA, etc.)",
  'Other, please specify']}

Education: only the apostrophe changes (`’` in 2025, `'` in 2026) and the text of "Other". Age: "Under 18" only exists in 2026.

In [5]:
compare_answers("DevType", "Role")

{'only in 2025': ['Founder, technology or otherwise',
  'Other (please specify):',
  'UX, Research Ops or UI design professional'],
 'only in 2026': ['Developer advocate or developer relations professional',
  'Educator or instructor',
  'Founder or entrepreneur',
  'Hardware engineer',
  'Marketing/Sales professional',
  'Other',
  'Platform engineer',
  'Site reliability engineer',
  'Software engineer',
  'Solutions engineer, implementation engineer, or forward-deployed engineer',
  'Technical writer or documentation professional',
  'UX, research, or UI design professional']}

Roles: 12 roles are new in 2026 (for example "Software engineer"). Detailed roles are not comparable, but the **role families** are.

In [6]:
langs_25 = set(r25["LanguageHaveWorkedWith"].dropna().str.split(";").explode())
langs_26 = set(r26["LanguagesHaveWorkedWith"].dropna().str.split(";").explode())
print("Languages only in 2026:", sorted(langs_26 - langs_25))
print("Languages only in 2025:", sorted(langs_25 - langs_26))

Languages only in 2026: ['Clojure', 'Haskell', 'Julia', 'Nix', 'Objective-C', 'Other, please specify', 'Solidity']
Languages only in 2025: []


In [7]:
compare_answers("Industry", "Industry")

{'only in 2025': ['Banking/Financial Services',
  'Computer Systems Design and Services',
  'Higher Education',
  'Internet, Telecomm or Information Services',
  'Media & Advertising Services',
  'Other:',
  'Retail and Consumer Services',
  'Software Development',
  'Transportation, or Supply Chain'],
 'only in 2026': ['Agriculture, forestry, fishing, and hunting',
  'Banking/financial services',
  'Broadcasting or publishing',
  'Education',
  'Engineering services or applied R&D',
  'Hospitality or travel',
  'IT services, consulting, systems integration, or managed services',
  'Life sciences or biotechnology',
  'Media or advertising services',
  'Mining or construction',
  'Non-profit or social services',
  'Other, please specify',
  'Retail, e-commerce, or consumer services',
  'Scientific research',
  'Software development',
  'Transportation or supply chain',
  'Utilities, internet, or telecommunications']}

Industry: the list of answers changed a lot, it is **not** compared.

## 2. Salary: same question, same cleaning, own exchange rate

In [8]:
from src.cleaning import clean_survey
from src.population import paid_population, focus_country, comparison_group
from src.analysis import evolution_of_groups

paid = {year: paid_population(clean_survey(year)) for year in [2025, 2026]}
evolution_of_groups({
    "France": (focus_country(paid[2025]), focus_country(paid[2026])),
    "Rest of the EU": (comparison_group(paid[2025], "European Union (without France)"),
                       comparison_group(paid[2026], "European Union (without France)")),
})

,group,n_2025,median_2025,n_2026,median_2026,change_percent,reliable
0,France,920,55000.0,504,55000.0,0.000000,True
1,Rest of the EU,6907,64440.0,3922,66521.0,3.229361,True


Each year is converted to euros with the rate used by Stack Overflow that year. Salaries are **nominal** (not corrected for inflation) and the respondents are not the same people each year: the comparison describes two samples of volunteers, not the evolution of the same people.